In [1]:
%pip install pyspark

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("OlistDataCleaning") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

In [3]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

customers_path = "customers_final_20260331_232800.csv"
geolocation_path = "geolocation_final_20260331_233312.csv"

customers_df = spark.read.csv(customers_path, header=True, inferSchema=True)
geolocation_df = spark.read.csv(geolocation_path, header=True, inferSchema=True)

# Create an index per zip for both tables and map customers to geolocations by modulo
geo_w = Window.partitionBy("geolocation_zip_code_prefix").orderBy(F.monotonically_increasing_id())
geo_indexed = (
    geolocation_df
    .withColumn("geo_index", F.row_number().over(geo_w))
    .withColumn("geo_count", F.count("*").over(Window.partitionBy("geolocation_zip_code_prefix")))
    .withColumnRenamed("geolocation_zip_code_prefix", "customer_zip_code_prefix")
)

geo_counts = geo_indexed.select("customer_zip_code_prefix", "geo_count").dropDuplicates(["customer_zip_code_prefix"])

cust_w = Window.partitionBy("customer_zip_code_prefix").orderBy(F.monotonically_increasing_id())
customers_indexed = customers_df.withColumn("cust_index", F.row_number().over(cust_w))

customers_with_geo_count = customers_indexed.join(
    geo_counts,
    on="customer_zip_code_prefix",
    how="inner"
)

customers_with_geo_index = customers_with_geo_count.withColumn(
    "geo_index",
    F.pmod(F.col("cust_index") - F.lit(1), F.col("geo_count")) + F.lit(1)
)

joined_df = customers_with_geo_index.join(
    geo_indexed,
    on=["customer_zip_code_prefix", "geo_index"],
    how="inner"
)

joined_df = joined_df.drop("geo_count", "geo_index", "cust_index")

joined_df.show(5)
print(f"Total rows: {joined_df.count()}")

+------------------------+--------------------+--------------------+-------------+-------------------+-------------------+
|customer_zip_code_prefix|         customer_id|  customer_unique_id|customer_city|    geolocation_lat|    geolocation_lng|
+------------------------+--------------------+--------------------+-------------+-------------------+-------------------+
|                    1226|a32e9f128bf594a9f...|04964ca17488b7612...|    SAO PAULO|-23.538190850683794|-46.651323227306854|
|                    1238|89aac25164a5f84e4...|3ac427a63b9d1586f...|    SAO PAULO| -23.54195269441404| -46.65931762155928|
|                    2531|6c2d4f0f802bfa13d...|18256891fed9a64ad...|    SAO PAULO|-23.498286024150236| -46.65922176395056|
|                    2542|6f8c1f9a45ed437e3...|e1b207cdbe1a6f581...|    SAO PAULO|-23.490407935917897|-46.658786595727854|
|                    3359|cdb99897893ebbeb4...|cb4418cdf55766fec...|    SAO PAULO|-23.565371310531198| -46.54729240747858|
+---------------

In [4]:
import csv
import os
import pandas as pd
from datetime import datetime

joined_pd = joined_df.toPandas()
output_dir = os.getcwd()

output_path = os.path.join(
    output_dir, f"customers_and_geolocation_final_{datetime.now():%Y%m%d_%H%M%S}.csv"
 )
joined_pd.to_csv(output_path, index=False, quoting=csv.QUOTE_ALL)
print(f"CSV salvo em: {output_path}")

CSV salvo em: c:\Users\Sofhia\Downloads\archive\customers_and_geolocation_final_20260403_135844.csv
